# Chapter 10 — Distributed Data Parallel Training

Training GPT-3 (175B parameters) on a single GPU would take decades. Modern LLM training requires distributing computation across hundreds or thousands of GPUs working in concert. This chapter covers the parallelism strategies that make large-scale training possible, from the straightforward to the highly sophisticated.

In [ ]:
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler

def train(rank, world_size):
    # Initialize process group
    dist.init_process_group("nccl", rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)

    # Wrap model with DDP
    model = GPT(config).cuda(rank)
    model = DDP(model, device_ids=[rank])

    # Distributed sampler ensures each GPU gets different data
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank)
    dataloader = DataLoader(dataset, sampler=sampler, batch_size=32)

    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

    for epoch in range(num_epochs):
        sampler.set_epoch(epoch)  # Shuffle differently each epoch
        for batch in dataloader:
            loss = model(batch.cuda(rank)).loss
            loss.backward()       # Gradients auto-synced by DDP
            optimizer.step()
            optimizer.zero_grad()

    dist.destroy_process_group()

---

**Course:** Aprenda LLM | **Chapter 10**

This notebook is part of the [Aprenda LLM](https://magestic.dev) course.